# 🎬 Nik Studio — Colab GPU Worker

This notebook generates your scene images on a free Colab GPU.

**How it works**

| Nik Studio (your PC) | Google Drive | Colab (here) |
| --- | --- | --- |
| 🚀 Render Episode | → `Jobs/Scene01.json` | picks up the job, generates |
| 📥 Import Results | ← `Results/Scene01.png` | saves the image |

**Before you run anything**

1. Set **Runtime → Change runtime type → GPU** (a free T4 is plenty).
2. In Nik Studio, the episode's `episode.json` needs `"backend": "Colab"`.
3. Press 🚀 **RENDER EPISODE** (or 🎬 Render Scene) first, so there are
   jobs waiting.
4. Let Google Drive finish syncing.

Then run the three cells below in order.

**Running this a second time?** Choose **Runtime → Restart session**
first. The model from the last run stays on the GPU otherwise, and
the next one has nowhere to go.


In [ ]:
# 1. Install the image generation packages (takes about a minute)
#
# Nothing is pinned. Pinning diffusers to an older release looked like a
# fix for IP-Adapter, but 0.31 cannot even import against the transformers
# Colab ships now ("cannot import name 'FLAX_WEIGHTS_NAME'"), and pinning
# transformers as well only moves the fight elsewhere.
!pip install -q diffusers transformers accelerate safetensors
print("packages installed")


In [ ]:
# 2. Connect Google Drive and find the folder that has jobs waiting

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Anywhere under NikStudio that contains a Jobs folder counts, so this
# works whether you share a whole episode or only a sync_folder.
SEARCH_ROOT = Path('/content/drive/MyDrive/NikStudio')

found = []

if SEARCH_ROOT.exists():
    for jobs in sorted(SEARCH_ROOT.rglob('Jobs')):
        if jobs.is_dir():
            waiting = len(list(jobs.glob('*.json')))
            found.append((jobs.parent, waiting))

with_work = [(folder, n) for folder, n in found if n]

if len(with_work) == 1:
    # Exactly one folder has work, so choose it. The next cell picks this
    # up, which saves editing a path by hand - the step that goes wrong
    # most often.
    EPISODE = str(with_work[0][0])
    print(f"Using: {EPISODE}")
    print(f"       {with_work[0][1]} job(s) waiting\n")
    print("Run the next cell.")

elif len(with_work) > 1:
    print("More than one folder has jobs waiting:\n")
    for folder, n in with_work:
        print(f"  {folder}   ({n} job(s))")
    print("\nSet EPISODE at the top of the next cell to the one you want.")

else:
    print(f"No jobs waiting under {SEARCH_ROOT}\n")
    if found:
        print("Folders found, but none have jobs in them:")
        for folder, n in found:
            print(f"  {folder}")
        print()
    print("Things to check:")
    print("  - did you press Render Scene / Render Episode in Nik Studio?")
    print("  - has Google Drive finished syncing?")
    print("  - is this what Drive actually contains:")
    for p in sorted(Path('/content/drive/MyDrive').glob('*'))[:20]:
        print("      ", p)


In [ ]:
# 3. Run. The folder comes from the cell above.

import json
import time
from pathlib import Path

# ----------------------------------------------------------------------
# SETTINGS
# ----------------------------------------------------------------------

# The folder Nik Studio and Colab share. In the notebook the previous cell
# finds this for you and sets it, so there is usually nothing to change
# here - globals() is checked first precisely so that value is not
# overwritten when this cell runs.
EPISODE = globals().get("EPISODE") or (
    "/content/drive/MyDrive/NikStudio/Exchange/Bath Time Song"
)

# How long to keep watching for new jobs, in minutes.
# Set to 0 to process the jobs that exist right now and then stop.
WATCH_MINUTES = 60

# How often to look for new jobs, in seconds.
POLL_SECONDS = 10


# ----------------------------------------------------------------------

def mount_drive():
    """Mount Google Drive. Does nothing when run outside Colab."""

    try:
        from google.colab import drive
    except ImportError:
        print("Not running in Colab - skipping Drive mount.")
        return

    if Path("/content/drive").exists():
        print("Drive already mounted.")
        return

    drive.mount("/content/drive")


# ----------------------------------------------------------------------

class Worker:

    def __init__(self, episode):

        self.episode = Path(episode)
        self.jobs = self.episode / "Jobs"
        self.results = self.episode / "Results"

        self.pipe = None
        self.loaded_model = None
        self.has_adapter = False

        # Turned on only after an out of memory failure. CPU offload and
        # IP-Adapter together are what produced "'tuple' object has no
        # attribute 'shape'", so it is no longer the default.
        self.force_offload = False

    # ------------------------------------------------------------------

    def check_folders(self):

        if not self.episode.exists():
            raise SystemExit(
                f"Folder not found:\n  {self.episode}\n\n"
                "Run the previous cell again - it lists the folders that "
                "have jobs waiting and sets this one for you.\n"
                "If you are running this script on its own, edit EPISODE "
                "at the top."
            )

        self.jobs.mkdir(parents=True, exist_ok=True)
        self.results.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------

    def report_gpu(self):

        import torch

        print("Torch Version  :", torch.__version__)
        print("CUDA Available :", torch.cuda.is_available())

        if torch.cuda.is_available():
            print("GPU            :", torch.cuda.get_device_name(0))
            print("VRAM           :", self.vram_report())
        else:
            print(
                "\n⚠ No GPU. In Colab choose "
                "Runtime > Change runtime type > GPU, then run again."
            )

    # ------------------------------------------------------------------

    def load_model(self, model, with_reference=False):
        """
        Load the model once and reuse it for every job.

        `with_reference` also loads IP-Adapter, which lets a picture of the
        character steer the generation. A written description alone only
        gets a character roughly right; the picture is what keeps the same
        face from scene to scene.
        """

        if (
            self.pipe is not None
            and self.loaded_model == model
            and self.has_adapter == with_reference
        ):
            return self.pipe

        import gc

        import torch
        from diffusers import AutoPipelineForText2Image

        # Let go of the previous pipeline BEFORE building a new one.
        # Without this the old model is still on the card while the new
        # one loads, which needs twice the memory and fails on a T4.
        self.release()

        print(f"\nLoading model : {model}")

        use_gpu = torch.cuda.is_available()

        pipe = AutoPipelineForText2Image.from_pretrained(
            model,
            torch_dtype=torch.float16 if use_gpu else torch.float32,
            variant="fp16" if use_gpu else None,
        )

        # IP-Adapter is loaded before the model is placed, because CPU
        # offload has to be the last thing set up.
        if with_reference:
            try:
                pipe.load_ip_adapter(
                    "h94/IP-Adapter",
                    subfolder="sdxl_models",
                    weight_name="ip-adapter_sdxl.bin",
                )
                print("IP-Adapter loaded - using your character reference.")
            except Exception as error:
                print(f"⚠ Could not load IP-Adapter: {error}")
                print("  Carrying on with the text description only.")
                with_reference = False

        if use_gpu and self.force_offload:
            # Offloading keeps most of the model in system RAM and moves
            # each piece onto the card as it is needed: slower per image,
            # but it finishes instead of running out.
            #
            # Only used after an out of memory failure, because offload
            # and IP-Adapter together break generation on a T4.
            pipe.enable_model_cpu_offload()
            print("Low VRAM mode - slower, but it fits.")
        else:
            pipe = pipe.to("cuda" if use_gpu else "cpu")

        # VAE slicing cuts the peak memory of the decode step for almost
        # no cost, and does not touch attention. Newer diffusers moved it
        # onto the vae itself.
        try:
            pipe.vae.enable_slicing()
        except AttributeError:
            try:
                pipe.enable_vae_slicing()
            except (AttributeError, TypeError):
                pass

        # Attention slicing, however, replaces the UNet's attention
        # processors - and IP-Adapter needs its own. Enabling both is what
        # produced "'tuple' object has no attribute 'shape'" during
        # generation, and then "SlicedAttnProcessor.__init__() missing
        # slice_size" when the adapter tried to come back out.
        #
        # So it is only used when no character reference is involved.
        if not with_reference:
            try:
                pipe.enable_attention_slicing()
            except (AttributeError, TypeError):
                pass

        self.pipe = pipe
        self.loaded_model = model
        self.has_adapter = with_reference

        print("Model ready.\n")

        return pipe

    # ------------------------------------------------------------------

    def vram_report(self):
        """
        Free and total VRAM, so memory trouble is visible early.

        This is a status line and nothing more, so it never raises. It
        once turned a finished image into a reported failure, which is
        exactly what a diagnostic must not do.
        """

        try:
            import torch

            if not torch.cuda.is_available():
                return "no GPU"

            free, total = torch.cuda.mem_get_info()

            gb = 1024 ** 3

            return f"{free / gb:.1f}GB free of {total / gb:.1f}GB"

        except Exception:
            return "unknown"

    def low_vram(self):
        """
        True when the card is too small to hold the whole model.

        A free Colab T4 has about 15GB, which SDXL and IP-Adapter together
        overrun. Anything from 24GB up runs them outright.
        """

        try:
            import torch

            if not torch.cuda.is_available():
                return False

            total = torch.cuda.get_device_properties(0).total_memory

            return total < 20 * 1024 ** 3

        except Exception:
            return False

    def reclaim(self):
        """
        Recover VRAM held by an earlier run of this cell.

        Running the cell twice leaves the previous pipeline in the
        notebook's memory, and the card starts full: "VRAM: 0.0GB free of
        14.6GB". Nothing this worker does can rescue that - the model
        belongs to an object it cannot see - so the honest thing is to
        try, and then say plainly what to do.
        """

        try:
            import gc

            import torch

            if not torch.cuda.is_available():
                return

            free, total = torch.cuda.mem_get_info()

            gb = 1024 ** 3

            # Under 4GB free is not enough for SDXL to decode, however
            # much of the model has been offloaded.
            if free > 4 * gb:
                return

            print("\n⚠ The GPU is nearly full before we have even started.")
            print("  Trying to reclaim it...")

            gc.collect()
            torch.cuda.empty_cache()

            free, _ = torch.cuda.mem_get_info()

            print(f"  {self.vram_report()}")

            if free > 4 * gb:
                print("  Recovered enough to carry on.\n")
                return

            raise SystemExit(
                "\nThe GPU is still full, and it is held by something this "
                "cell cannot reach - almost always a model left behind by "
                "an earlier run.\n\n"
                "In Colab choose  Runtime > Restart session,  then run the "
                "cells again from the top.\n"
                "Nothing is lost: images already generated stay in Drive."
            )

        except SystemExit:
            raise

        except Exception:
            # Never let a memory check stop a run that might have worked.
            return

    # ------------------------------------------------------------------

    def release(self):
        """Give the card its memory back. Never raises."""

        if self.pipe is None:
            return

        import gc

        try:
            import torch
        except ImportError:
            torch = None

        self.pipe = None
        self.loaded_model = None
        self.has_adapter = False

        gc.collect()

        try:
            if torch is not None and torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    # ------------------------------------------------------------------

    def pending_jobs(self):
        """
        Job files that still need doing.

        A job whose result file already exists is skipped, so re-running
        this script never regenerates work that is already finished.
        """

        pending = []

        for job_file in sorted(self.jobs.glob("*.json")):

            try:
                job = json.loads(job_file.read_text(encoding="utf-8-sig"))
            except (OSError, ValueError) as error:
                print(f"⚠ Skipping unreadable job {job_file.name}: {error}")
                continue

            output = self.episode / job.get(
                "output",
                f"Results/{job.get('scene', job_file.stem)}.png",
            )

            if output.exists() and output.stat().st_size > 0:
                continue

            pending.append((job_file, job))

        return pending

    # ------------------------------------------------------------------

    def run_job(self, job_file, job):

        scene = job.get("scene", job_file.stem)
        prompt = job.get("prompt", "")

        if not prompt.strip():
            print(f"⚠ {scene}: job has no prompt, skipping.")
            return False

        model = job.get("model") or "stabilityai/sdxl-turbo"

        # Nik Studio may write a short name; expand it to a real repo id.
        if "/" not in model:
            model = {
                "sdxl-turbo": "stabilityai/sdxl-turbo",
                "sdxl": "stabilityai/stable-diffusion-xl-base-1.0",
                "flux": "black-forest-labs/FLUX.1-schnell",
            }.get(model, model)

        # Character reference pictures, if the job carries any.
        #
        # "use_reference": false in the job turns them off. IP-Adapter is
        # the least stable part of this pipeline; being able to switch it
        # off without editing anything is worth having.
        references = []

        for name in (
            job.get("reference_images", [])
            if job.get("use_reference", True)
            else []
        ) or []:

            path = self.episode / name

            if path.exists():
                references.append(path)
            else:
                print(f"⚠ reference not found, skipping: {name}")

        pipe = self.load_model(model, with_reference=bool(references))

        print(f"🎨 {scene} : {prompt[:70]}...")

        guidance = float(job.get("guidance", 0.0))

        kwargs = {
            "prompt": prompt,
            "num_inference_steps": int(job.get("steps", 4)),
            "guidance_scale": guidance,
            "width": int(job.get("width", 1024)),
            "height": int(job.get("height", 1024)),
        }

        # A negative prompt needs classifier free guidance to do anything.
        # Distilled models such as SDXL-Turbo run at guidance 0, where it
        # is ignored at best and an error at worst.
        negative = job.get("negative_prompt", "")

        if negative and guidance > 1:
            kwargs["negative_prompt"] = negative

        if references and self.has_adapter:

            from PIL import Image

            images = [Image.open(p).convert("RGB") for p in references]

            kwargs["ip_adapter_image"] = (
                images[0] if len(images) == 1 else images
            )

            pipe.set_ip_adapter_scale(
                float(job.get("reference_strength", 0.6))
            )

            print(f"   using {len(images)} character reference(s)")

        seed = int(job.get("seed", -1))

        if seed >= 0:
            import torch

            device = "cuda" if torch.cuda.is_available() else "cpu"
            kwargs["generator"] = torch.Generator(device).manual_seed(seed)

        started = time.time()

        try:
            image = pipe(**kwargs).images[0]

        except Exception as error:

            # IP-Adapter and diffusers fall out of step from time to time
            # ("'tuple' object has no attribute 'shape'" is the usual
            # shape of it). Losing the likeness is much better than
            # losing the image, so try once more without the reference.
            if "ip_adapter_image" not in kwargs:
                raise

            # Running out of memory is not the reference's fault. Dropping
            # it here would cost the likeness for nothing - let the caller
            # retry with offloading instead.
            if "out of memory" in str(error).lower():
                raise

            print(f"⚠ the character reference failed: {error}")
            print("   generating from the description alone instead")

            kwargs.pop("ip_adapter_image")

            # Dropping the argument is not enough. Loading the adapter
            # rewires the UNet - it sets encoder_hid_dim_type to
            # 'ip_image_proj' and then demands image embeds on every call.
            # The adapter has to come back out, or the retry fails with
            # "requires the keyword argument `image_embeds`".
            try:
                pipe.unload_ip_adapter()
                self.has_adapter = False
                print("   character reference unloaded")

            except Exception as unload_error:

                # A half removed adapter leaves the UNet in a state that
                # fails every later job with "'NoneType' object has no
                # attribute 'image_projection_layers'". Rebuilding is
                # slower than carrying on, and the only thing that works.
                print(f"   could not unload the adapter: {unload_error}")
                print("   rebuilding the pipeline without it")

                self.release()

                pipe = self.load_model(model, with_reference=False)

            image = pipe(**kwargs).images[0]

        output = self.episode / job.get("output", f"Results/{scene}.png")
        output.parent.mkdir(parents=True, exist_ok=True)

        # Write to a temporary name first, then rename. Nik Studio watches
        # this folder, and must never pick up a half written image.
        temp = output.with_suffix(output.suffix + ".part")

        # PIL picks the format from the file extension, and this temporary
        # name ends in ".part", so the format has to be named outright.
        image_format = {
            ".png": "PNG",
            ".jpg": "JPEG",
            ".jpeg": "JPEG",
            ".webp": "WEBP",
        }.get(output.suffix.lower(), "PNG")

        image.save(temp, format=image_format)
        temp.replace(output)

        print(
            f"✅ {scene} done in {time.time() - started:.1f}s "
            f"-> {output.relative_to(self.episode)}"
        )
        print(f"   VRAM: {self.vram_report()}")

        return True

    # ------------------------------------------------------------------

    def run(self, watch_minutes=WATCH_MINUTES, poll_seconds=POLL_SECONDS):

        print(f"Folder : {self.episode}")

        self.check_folders()
        self.report_gpu()
        self.reclaim()

        print(f"\nWatching : {self.jobs}")

        deadline = time.time() + watch_minutes * 60
        done = 0

        # A job that fails is not retried on the next pass. Without this
        # the watch loop regenerates the same broken job every few
        # seconds, burning GPU time for nothing.
        failed = set()

        while True:

            jobs = [
                (f, j) for f, j in self.pending_jobs()
                if f.name not in failed
            ]

            for job_file, job in jobs:

                try:
                    if self.run_job(job_file, job):
                        done += 1

                except Exception as error:

                    # Out of memory is worth one more go: drop everything,
                    # switch offloading on, and rebuild. It is slower, but
                    # it is the difference between an image and nothing.
                    if (
                        "out of memory" in str(error).lower()
                        and not self.force_offload
                    ):
                        print(f"⚠ {job_file.stem}: out of GPU memory")
                        print("   retrying with low VRAM mode")

                        self.release()
                        self.force_offload = True

                        try:
                            if self.run_job(job_file, job):
                                done += 1
                            continue

                        except Exception as retry_error:
                            error = retry_error

                    print(f"❌ {job_file.stem} failed: {error}")
                    print("   not retrying it in this run")
                    failed.add(job_file.name)

                    if "out of memory" in str(error).lower():
                        self.release()
                        print("   released the GPU before the next job")

            if time.time() >= deadline:
                break

            if not jobs:
                print("… waiting for jobs", end="\r")

            time.sleep(poll_seconds)

        # The notebook often stays open afterwards. Holding several GB of
        # a free GPU for nothing is rude to the next cell and to Colab.
        self.release()

        print(f"\nFinished. {done} image(s) generated.")
        print(f"GPU released. {self.vram_report()}")

        if failed:
            print(f"{len(failed)} job(s) failed: "
                  + ", ".join(sorted(f.replace('.json', '') for f in failed)))

        if done:
            print("Now press 📥 Import Results in Nik Studio.")


# ----------------------------------------------------------------------

if __name__ == "__main__":

    mount_drive()

    Worker(EPISODE).run()
